In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent   # /mnt/primary/Main
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from loader.load import load_aligned_plans_and_runs

from uncertainty_prediction.src import *
from uncertainty_prediction.config import *

from uncertainty_prediction.baselines.predictive.tlstm.data import *
from uncertainty_prediction.baselines.predictive.tlstm.model import *
from uncertainty_prediction.baselines.predictive.tlstm.util import *

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.utils.data import DataLoader
import torch.nn.functional as F

set_seed(42)

In [2]:

"""
Load and extract query plans and features
"""
# any set of plans is fine here (... assuming that plans are identical across runs)
queries_dir = "/mnt/lakehouse-raw-results/tpcds/lakehouse-a/20260222-191819Z/queries"

plans_by_query, runs_by_query, common = load_aligned_plans_and_runs(
    queries_dir=queries_dir,
    run_ids=RUN_IDS,
    collection=COLLECTION_NAME,
    schema=SCHEMA_NAME,
    instance=LAKEHOUSE_INSTANCE_NAME,
    metric=METRIC,
    xcol=XCOL,
    ycol=YCOL,
    parsed_results_root=PARSED_RESULTS_ROOT,
    canon_fn=canon_qid,
    min_runs=1,
    min_points_per_run=2,
    require_cols=(XCOL, YCOL),
)

In [3]:

"""
Split into training and testing sets
"""

train_qids, test_qids = split_query_ids(
    common,
    seed=SEED,
    test_frac=TEST_FRAC,
)
print("n_train:", len(train_qids), "n_test:", len(test_qids))

runs_train, runs_test = make_train_test_run_split(
    runs_by_query,
    train_qids=train_qids,
    test_qids=test_qids,
)


n_train: 332 n_test: 83


In [4]:

"""
Generate training and testing sets 
"""

tlstm_data = get_train_test_datasets_from_trino(
    plans_by_query=plans_by_query,
    runs_by_query=runs_by_query,
    train_qids=train_qids,
    test_qids=test_qids,
    xcol=XCOL,              # e.g. "t_rel_s"
    runtime_mode="mean",    # mean runtime across repeated runs
    batch_size=64,          # number of queries merged into one TLSTM batch
    condition_max_num=8,    # number of tokens per condition stream per node
    condition_op_dim=16,    # size of each token in condition1/condition2
    extra_dim=16,           # size of extra_infos vector
    sample_dim=32,          # size of sample branch vector
)

train_dataset = tlstm_data["train_dataset"]
test_dataset = tlstm_data["test_dataset"]

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=tlstm_collate_fn_identity,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=tlstm_collate_fn_identity,
)

In [5]:

"""
Build model
"""

device = "cuda" if torch.cuda.is_available() else "cpu"

model = TLSTMHeteroRuntimeRegressor(
    operator_dim=tlstm_data["operator_dim"],
    extra_dim=tlstm_data["extra_dim"],
    condition_op_dim=tlstm_data["condition_op_dim"],
    sample_dim=tlstm_data["sample_dim"],
    hidden_dim=128,
    hid_dim=256,
    head_dim=256,
    hetero_hidden_dim=64,
    log_sigma_min=-3.0,
    log_sigma_max=3.0,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)

In [6]:
import numpy as np

num_epochs = 30
best_val_loss = float("inf")
best_state = None

for epoch in range(num_epochs):
    model.train()
    train_losses = []

    for batch in train_loader:
        operators = batch["operators"].to(device)
        extra_infos = batch["extra_infos"].to(device)
        condition1s = batch["condition1s"].to(device)
        condition2s = batch["condition2s"].to(device)
        samples = batch["samples"].to(device)

        # already [L, N, 1] from the updated data.py
        condition_masks = batch["condition_masks"].to(device)

        mapping = batch["mapping"].to(device)

        # raw log-runtime target
        targets = batch["target_log_runtime"].to(device).reshape(-1)

        optimizer.zero_grad()

        mu_log, log_sigma, z = model(
            operators,
            extra_infos,
            condition1s,
            condition2s,
            samples,
            condition_masks,
            mapping,
        )

        loss = gaussian_nll_from_log_sigma(
            mu_log=mu_log,
            log_sigma=log_sigma,
            target_log_runtime=targets,
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_losses.append(loss.item())

    model.eval()
    val_losses = []

    with torch.no_grad():
        for batch in test_loader:
            operators = batch["operators"].to(device)
            extra_infos = batch["extra_infos"].to(device)
            condition1s = batch["condition1s"].to(device)
            condition2s = batch["condition2s"].to(device)
            samples = batch["samples"].to(device)
            condition_masks = batch["condition_masks"].to(device)
            mapping = batch["mapping"].to(device)
            targets = batch["target_log_runtime"].to(device).reshape(-1)

            mu_log, log_sigma, z = model(
                operators,
                extra_infos,
                condition1s,
                condition2s,
                samples,
                condition_masks,
                mapping,
            )

            loss = gaussian_nll_from_log_sigma(
                mu_log=mu_log,
                log_sigma=log_sigma,
                target_log_runtime=targets,
            )
            val_losses.append(loss.item())

    mean_train = float(np.mean(train_losses))
    mean_val = float(np.mean(val_losses))
    print(f"Epoch {epoch+1:03d} | train_nll={mean_train:.6f} | val_nll={mean_val:.6f}")

    if mean_val < best_val_loss:
        best_val_loss = mean_val
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)

/opt/miniconda3/lib/python3.10/site-packages/torch/nn/modules/linear.py:125: UserWarning: Could not parse CUBLAS_WORKSPACE_CONFIG, using default workspace size of 8519680 bytes. (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:144.)
  return F.linear(input, self.weight, self.bias)
/opt/miniconda3/lib/python3.10/site-packages/torch/nn/modules/linear.py:125: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:217.)
  return F.line

Epoch 001 | train_nll=2.695477 | val_nll=3.051927
Epoch 002 | train_nll=1.060770 | val_nll=2.467761
Epoch 003 | train_nll=1.051109 | val_nll=2.147435
Epoch 004 | train_nll=1.031638 | val_nll=1.928869
Epoch 005 | train_nll=1.041612 | val_nll=1.769467
Epoch 006 | train_nll=1.026553 | val_nll=1.693431
Epoch 007 | train_nll=1.011100 | val_nll=1.595931
Epoch 008 | train_nll=1.004024 | val_nll=1.491780
Epoch 009 | train_nll=1.014142 | val_nll=1.343868
Epoch 010 | train_nll=0.996757 | val_nll=1.386498
Epoch 011 | train_nll=1.006405 | val_nll=1.333427
Epoch 012 | train_nll=0.992128 | val_nll=1.174793
Epoch 013 | train_nll=0.998772 | val_nll=1.193879
Epoch 014 | train_nll=1.004520 | val_nll=1.020013
Epoch 015 | train_nll=1.008100 | val_nll=1.048797
Epoch 016 | train_nll=1.013015 | val_nll=1.135693
Epoch 017 | train_nll=1.020093 | val_nll=1.018020
Epoch 018 | train_nll=0.988652 | val_nll=1.024485
Epoch 019 | train_nll=0.999072 | val_nll=1.116004
Epoch 020 | train_nll=0.993203 | val_nll=1.064838


In [7]:
import numpy as np
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def qerror_numpy(y_pred, y_true, eps=1e-12):
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    return np.maximum((y_pred + eps) / (y_true + eps), (y_true + eps) / (y_pred + eps))


all_mu_log = []
all_sigma_log = []
all_targets_log = []
all_qids = []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        mu_log, log_sigma, z = model(
            batch["operators"].to(device),
            batch["extra_infos"].to(device),
            batch["condition1s"].to(device),
            batch["condition2s"].to(device),
            batch["samples"].to(device),
            batch["condition_masks"].to(device),   # no unsqueeze
            batch["mapping"].to(device),
        )

        sigma_log = torch.exp(log_sigma)

        all_mu_log.append(mu_log.cpu().numpy().reshape(-1))
        all_sigma_log.append(sigma_log.cpu().numpy().reshape(-1))
        all_targets_log.append(batch["target_log_runtime"].cpu().numpy().reshape(-1))
        all_qids.extend(batch["qids"])

mu_log = np.concatenate(all_mu_log)
sigma_log = np.concatenate(all_sigma_log)
targets_log = np.concatenate(all_targets_log)

# convert back to raw runtime space
preds = np.exp(mu_log)
targets = np.exp(targets_log)

qerr = qerror_numpy(preds, targets)
lower = np.exp(mu_log - 1.96 * sigma_log)
upper = np.exp(mu_log + 1.96 * sigma_log)

metrics = {
    "mae_log": float(mean_absolute_error(targets_log, mu_log)),
    "rmse_log": float(np.sqrt(mean_squared_error(targets_log, mu_log))),
    "mae_runtime": float(mean_absolute_error(targets, preds)),
    "rmse_runtime": float(np.sqrt(mean_squared_error(targets, preds))),
    "r2_log": float(r2_score(targets_log, mu_log)),
    "r2_runtime": float(r2_score(targets, preds)),
    "median_q_error": float(np.median(qerr)),
    "p95_q_error": float(np.percentile(qerr, 95)),
    "mean_q_error": float(np.mean(qerr)),
    "gmean_q_error": float(np.exp(np.mean(np.log(qerr + 1e-12)))),
    "mean_sigma_log": float(np.mean(sigma_log)),
    "median_sigma_log": float(np.median(sigma_log)),
    "coverage_95": float(np.mean((targets >= lower) & (targets <= upper))),
}

print(metrics)

{'mae_log': 1.381080150604248, 'rmse_log': 1.7149553197484535, 'mae_runtime': 16.02056312561035, 'rmse_runtime': 82.23024672565747, 'r2_log': 0.013535916805267334, 'r2_runtime': -0.033530473709106445, 'median_q_error': 3.327104907621021, 'p95_q_error': 23.211032475565375, 'mean_q_error': 12.341745877239026, 'gmean_q_error': 3.979197862007937, 'mean_sigma_log': 1.7941210269927979, 'median_sigma_log': 1.6779836416244507, 'coverage_95': 0.963855421686747}


In [8]:
from scipy.stats import spearmanr
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def print_metrics_for_excel(metrics):
    print("\t".join(str(v) for v in metrics.values()))

def print_metric_headers_for_excel(metrics):
    print("\t".join(metrics.keys()))
    
def evaluate_runtime_metrics(preds_unnorm, labels_unnorm, eps=1e-12):
    preds_unnorm = np.asarray(preds_unnorm, dtype=float).reshape(-1)
    labels_unnorm = np.asarray(labels_unnorm, dtype=float).reshape(-1)

    if preds_unnorm.shape[0] != labels_unnorm.shape[0]:
        raise ValueError("preds_unnorm and labels_unnorm must have the same length")

    qerr = qerror_numpy(preds_unnorm, labels_unnorm)

    spearman = spearmanr(labels_unnorm, preds_unnorm).statistic
    if spearman is None or np.isnan(spearman):
        spearman = 0.0

    metrics = {
        "mae": float(mean_absolute_error(labels_unnorm, preds_unnorm)),
        "rmse": float(np.sqrt(mean_squared_error(labels_unnorm, preds_unnorm))),
        "r2": float(r2_score(labels_unnorm, preds_unnorm)),
        "spearman": float(spearman),
        "median_q_error": float(np.median(qerr)),
        "mean_q_error": float(np.mean(qerr)),
        "gmean_q_error": float(np.exp(np.mean(np.log(qerr + eps)))),
        "p90_q_error": float(np.percentile(qerr, 90)),
        "p95_q_error": float(np.percentile(qerr, 95)),
        "max_q_error": float(np.max(qerr)),
    }
    return metrics

test_metrics = evaluate_runtime_metrics(preds, targets)

print_metric_headers_for_excel(test_metrics)
print_metrics_for_excel(test_metrics)



mae	rmse	r2	spearman	median_q_error	mean_q_error	gmean_q_error	p90_q_error	p95_q_error	max_q_error
16.02056271196848	82.23024212490182	-0.03353024476836852	0.043418426570747126	3.327104907621021	12.341745877239026	3.979197862007937	12.58739819599013	23.211032475565375	513.2112227277437


In [9]:
import numpy as np

def print_metrics_for_excel(metrics):
    print("\t".join(str(v) for v in metrics.values()))

def collect_targets_from_loader(loader, target_key="target_runtime"):
    ys = []
    for batch in loader:
        y = batch[target_key]
        if isinstance(y, torch.Tensor):
            y = y.detach().cpu().numpy()
        ys.append(np.asarray(y).reshape(-1))
    return np.concatenate(ys)


y_train = collect_targets_from_loader(train_loader, target_key="target_runtime")
y_test = collect_targets_from_loader(test_loader, target_key="target_runtime")

baseline_mean_runtime = float(np.mean(y_train))
preds_test = np.full_like(y_test, baseline_mean_runtime, dtype=float)

baseline_metrics = evaluate_runtime_metrics(preds_test, y_test)
print_metric_headers_for_excel(baseline_metrics)
print_metrics_for_excel(baseline_metrics)

mae	rmse	r2	spearman	median_q_error	mean_q_error	gmean_q_error	p90_q_error	p95_q_error	max_q_error
33.62780158293535	81.56810819225987	-0.01695241773661671	0.0	26.595335917855408	39.38617594388417	17.727185806152804	87.45387890914255	156.78349245024944	342.52016296534646


/tmp/ipykernel_1569/3602145189.py:20: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman = spearmanr(labels_unnorm, preds_unnorm).statistic
